# Road Accident Severity Prediction — End-to-End ML Pipeline¶
Dataset: **India Road Accident Dataset – Predictive Analysis** (Kaggle, khushikyad001)

# Notebook 02: Data Preprocessing & Feature Engineering

1. Data Cleaning
2. Missing Value Treatment
3. Duplicate Removal
4. Feature Engineering
5. Feature Selection
6. Target Encoding
7. Train-Test Split
8. Data Preprocessing Pipeline
9. Saving Processed Data


In [1]:
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load Dataset

import os
import pandas as pd

csv_path = "../data/raw/accident_prediction_india.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at: {csv_path}")

df = pd.read_csv(csv_path)

# Normalize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# Rename columns to project schema
rename_dict = {
    'state_name': 'state',
    'city_name': 'city',
    'accident_severity': 'severity',
    'number_of_vehicles_involved': 'num_vehicles',
    'vehicle_type_involved': 'vehicle_type',
    'weather_conditions': 'weather',
    'speed_limits': 'speed_limit',
    'alcohol_involvement': 'alcohol_involved',
    'gender': 'driver_gender',
    'time_of_day': 'time'
}

df.rename(columns=rename_dict, inplace=True)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (3000, 22)


In [3]:
print("Dataset Information")

df.info()

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   state                      3000 non-null   str  
 1   city                       3000 non-null   str  
 2   year                       3000 non-null   int64
 3   month                      3000 non-null   str  
 4   day_of_week                3000 non-null   str  
 5   time                       3000 non-null   str  
 6   severity                   3000 non-null   str  
 7   num_vehicles               3000 non-null   int64
 8   vehicle_type               3000 non-null   str  
 9   number_of_casualties       3000 non-null   int64
 10  number_of_fatalities       3000 non-null   int64
 11  weather                    3000 non-null   str  
 12  road_type                  3000 non-null   str  
 13  road_condition             3000 non-null   str  
 14  lighting_condit

In [4]:
print("Summary Statistics")

display(df.describe(include='all').T)

Summary Statistics


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
state,3000,32,Goa,109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,3000,28,Unknown,2138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,3000.0,NaN,NaN,NaN,2020.53,1.683858,2018.0,2019.0,2021.0,2022.0,2023.0
month,3000,12,March,266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,3000,7,Wednesday,468,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time,3000,1263,3:40,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
severity,3000,3,Minor,1034,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_vehicles,3000.0,NaN,NaN,NaN,2.996,1.428285,1.0,2.0,3.0,4.0,5.0
vehicle_type,3000,7,Truck,449,NaN,NaN,NaN,NaN,NaN,NaN,NaN
number_of_casualties,3000.0,NaN,NaN,NaN,5.066,3.214097,0.0,2.0,5.0,8.0,10.0


In [5]:
print("Missing Values")

missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum()/len(df))*100
})

display(missing.sort_values("Percentage", ascending=False))

Missing Values


,Missing Values,Percentage
driver_license_status,975,32.500000
traffic_control_presence,716,23.866667
year,0,0.000000
month,0,0.000000
state,0,0.000000
city,0,0.000000
time,0,0.000000
day_of_week,0,0.000000
severity,0,0.000000
num_vehicles,0,0.000000


In [6]:
print("Missing values before imputation:\n")
print(df.isnull().sum())

# Driver Age
if 'driver_age' in df.columns:
    df['driver_age'].fillna(df['driver_age'].median(), inplace=True)

# Weather
if 'weather' in df.columns:
    df['weather'].fillna(df['weather'].mode()[0], inplace=True)

# Driver License Status
if 'driver_license_status' in df.columns:
    df['driver_license_status'].fillna(
        df['driver_license_status'].mode()[0],
        inplace=True
    )

# Traffic Control Presence
if 'traffic_control_presence' in df.columns:
    df['traffic_control_presence'].fillna(
        "Unknown",
        inplace=True
    )

print("\nMissing values after imputation:\n")
print(df.isnull().sum())

Missing values before imputation:

state                          0
city                           0
year                           0
month                          0
day_of_week                    0
time                           0
severity                       0
num_vehicles                   0
vehicle_type                   0
number_of_casualties           0
number_of_fatalities           0
weather                        0
road_type                      0
road_condition                 0
lighting_conditions            0
traffic_control_presence     716
speed_limit_(km/h)             0
driver_age                     0
driver_gender                  0
driver_license_status        975
alcohol_involved               0
accident_location_details      0
dtype: int64

Missing values after imputation:

state                          0
city                           0
year                           0
month                          0
day_of_week                    0
time                      

In [7]:
duplicates = df.duplicated().sum()

print(f"Duplicate Records : {duplicates}")

Duplicate Records : 0


In [8]:
df.drop_duplicates(inplace=True)

print("Dataset Shape After Removing Duplicates(If there are ANY)")

print(df.shape)

Dataset Shape After Removing Duplicates(If there are ANY)
(3000, 22)


In [9]:
print(df.dtypes)

state                          str
city                           str
year                         int64
month                          str
day_of_week                    str
time                           str
severity                       str
num_vehicles                 int64
vehicle_type                   str
number_of_casualties         int64
number_of_fatalities         int64
weather                        str
road_type                      str
road_condition                 str
lighting_conditions            str
traffic_control_presence       str
speed_limit_(km/h)           int64
driver_age                   int64
driver_gender                  str
driver_license_status          str
alcohol_involved               str
accident_location_details      str
dtype: object


In [10]:
# Feature Engineering

# 1. Extract Hour from Time
df['accident_hour'] = pd.to_datetime(
    df['time'],
    format='%H:%M',
    errors='coerce'
).dt.hour

# 2. Categorize Time of Day
def get_time_of_day(hour):
    if 0 <= hour < 5:
        return 'Late Night'
    elif 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    else:
        return 'Evening'
df['time_of_day'] = df['accident_hour'].apply(get_time_of_day)

# 3. Driver Age Group
def get_age_group(age):
    if age < 25:
        return 'Young'
    elif age <= 60:
        return 'Adult'
    else:
        return 'Senior'

df['driver_age_group'] = df['driver_age'].apply(get_age_group)
# 4. Multi-Vehicle Flag
df['is_multi_vehicle'] = df['num_vehicles'].apply(
    lambda x: 'Yes' if x > 1 else 'No'
)

# 5. Weekend Flag
df['is_weekend'] = df['day_of_week'].isin(
    ['Saturday', 'Sunday']
).map({True: 'Yes', False: 'No'})

# 6. Night Driving Flag
df['is_night'] = df['accident_hour'].apply(
    lambda x: 'Yes' if (x >= 20 or x < 6) else 'No'
)

# 7. Peak Hour Flag
df['is_peak_hour'] = df['accident_hour'].apply(
    lambda x: 'Yes' if (7 <= x <= 10) or (17 <= x <= 20) else 'No'
)
print("Feature Engineering Completed Successfully!\n")
display(
    df[
        [
            'severity',
            'time',
            'accident_hour',
            'time_of_day',
            'driver_age',
            'driver_age_group',
            'num_vehicles',
            'is_multi_vehicle',
            'day_of_week',
            'is_weekend',
            'is_night',
            'is_peak_hour'
        ]
    ].head()
)

Feature Engineering Completed Successfully!



,severity,time,accident_hour,time_of_day,driver_age,driver_age_group,num_vehicles,is_multi_vehicle,day_of_week,is_weekend,is_night,is_peak_hour
0,Serious,1:46,1,Late Night,66,Senior,5,Yes,Monday,No,Yes,No
1,Minor,21:30,21,Evening,60,Adult,5,Yes,Wednesday,No,Yes,No
2,Minor,5:37,5,Morning,26,Adult,5,Yes,Wednesday,No,Yes,No
3,Minor,0:31,0,Late Night,34,Adult,3,Yes,Saturday,Yes,Yes,No
4,Minor,11:21,11,Morning,30,Adult,5,Yes,Thursday,No,No,No


In [11]:
# Feature Selection

# Encode target variable
severity_map = {
    'Minor': 0,
    'Serious': 1,
    'Fatal': 2
}

df['target_severity'] = df['severity'].map(severity_map)

# Remove columns that should not be used for training
leakage_cols = [
    'id',
    'severity',
    'time',
    'number_of_casualties',
    'number_of_fatalities'
]

df_model = df.drop(columns=leakage_cols, errors='ignore')

print("Feature Selection Completed!")

print(f"\nNumber of Selected Features: {len(df_model.columns)}\n")

print("Selected Features:")

for col in df_model.columns:
    print(f"• {col}")

Feature Selection Completed!

Number of Selected Features: 26

Selected Features:
• state
• city
• year
• month
• day_of_week
• num_vehicles
• vehicle_type
• weather
• road_type
• road_condition
• lighting_conditions
• traffic_control_presence
• speed_limit_(km/h)
• driver_age
• driver_gender
• driver_license_status
• alcohol_involved
• accident_location_details
• accident_hour
• time_of_day
• driver_age_group
• is_multi_vehicle
• is_weekend
• is_night
• is_peak_hour
• target_severity


In [12]:
X = df_model.drop(columns=['target_severity'])
y = df_model['target_severity']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (3000, 25)
Target shape: (3000,)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Features : {X_train.shape}")
print(f"Testing Features  : {X_test.shape}")
print(f"Training Labels   : {y_train.shape}")
print(f"Testing Labels    : {y_test.shape}")

Training Features : (2400, 25)
Testing Features  : (600, 25)
Training Labels   : (2400,)
Testing Labels    : (600,)


In [14]:
# Identify feature types from TRAINING DATA
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

print("Numerical Features:")
print(numeric_cols)

print("\nCategorical Features:")
print(categorical_cols)

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Fit on training data and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Retrieve transformed feature names
encoded_cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)

all_features_transformed = numeric_cols + list(encoded_cat_names)

print(f"\nProcessed Training Shape : {X_train_processed.shape}")
print(f"Processed Testing Shape  : {X_test_processed.shape}")
print(f"Total Features After Encoding: {len(all_features_transformed)}")

Numerical Features:
['year', 'num_vehicles', 'speed_limit_(km/h)', 'driver_age']

Categorical Features:
['state', 'city', 'month', 'day_of_week', 'vehicle_type', 'weather', 'road_type', 'road_condition', 'lighting_conditions', 'traffic_control_presence', 'driver_gender', 'driver_license_status', 'alcohol_involved', 'accident_location_details', 'time_of_day', 'driver_age_group', 'is_multi_vehicle', 'is_weekend', 'is_night', 'is_peak_hour']

Processed Training Shape : (2400, 137)
Processed Testing Shape  : (600, 137)
Total Features After Encoding: 137


In [15]:
joblib.dump(
    all_features_transformed,
    "../models/feature_names.pkl"
)

print("Feature names saved successfully!")

Feature names saved successfully!


In [16]:
print("\nPreview of Processed Dataset:")
display(df_model.head())


Preview of Processed Dataset:


,state,city,year,month,day_of_week,num_vehicles,vehicle_type,weather,road_type,road_condition,...,alcohol_involved,accident_location_details,accident_hour,time_of_day,driver_age_group,is_multi_vehicle,is_weekend,is_night,is_peak_hour,target_severity
0,Jammu and Kashmir,Unknown,2021,May,Monday,5,Cycle,Hazy,National Highway,Wet,...,Yes,Curve,1,Late Night,Senior,Yes,No,Yes,No,1
1,Uttar Pradesh,Lucknow,2018,January,Wednesday,5,Truck,Hazy,Urban Road,Dry,...,Yes,Straight Road,21,Evening,Adult,Yes,No,Yes,No,0
2,Chhattisgarh,Unknown,2023,May,Wednesday,5,Pedestrian,Foggy,National Highway,Under Construction,...,No,Bridge,5,Morning,Adult,Yes,No,Yes,No,0
3,Uttar Pradesh,Lucknow,2020,June,Saturday,3,Bus,Rainy,State Highway,Dry,...,Yes,Straight Road,0,Late Night,Adult,Yes,Yes,Yes,No,0
4,Sikkim,Unknown,2021,August,Thursday,5,Cycle,Foggy,Urban Road,Wet,...,No,Intersection,11,Morning,Adult,Yes,No,No,No,0


In [17]:
# Create folders if they don't exist
os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# Save preprocessor
joblib.dump(preprocessor, "../models/preprocessor.pkl")

# Save processed datasets
joblib.dump(X_train_processed, "../data/processed/X_train.pkl")
joblib.dump(X_test_processed, "../data/processed/X_test.pkl")

joblib.dump(y_train, "../data/processed/y_train.pkl")
joblib.dump(y_test, "../data/processed/y_test.pkl")

print("All files saved successfully!")

All files saved successfully!


In [18]:
print(df[['severity']].head())

print("\nSeverity Distribution:")
print(df['severity'].value_counts())

print("\nPercentage Distribution:")
print(df['severity'].value_counts(normalize=True) * 100)

  severity
0  Serious
1    Minor
2    Minor
3    Minor
4    Minor

Severity Distribution:
severity
Minor      1034
Fatal       985
Serious     981
Name: count, dtype: int64

Percentage Distribution:
severity
Minor      34.466667
Fatal      32.833333
Serious    32.700000
Name: proportion, dtype: float64


In [19]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

In [20]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# Convert categorical columns to category codes
X_temp = X.copy()

for col in X_temp.select_dtypes(include="object").columns:
    X_temp[col] = X_temp[col].astype("category").cat.codes

mi = mutual_info_classif(
    X_temp,
    y,
    random_state=42
)

mi_scores = (
    pd.DataFrame({
        "Feature": X_temp.columns,
        "Mutual Information": mi
    })
    .sort_values("Mutual Information", ascending=False)
)

display(mi_scores)

,Feature,Mutual Information
3,month,0.032284
20,driver_age_group,0.016357
1,city,0.014996
8,road_type,0.014472
22,is_weekend,0.010322
15,driver_license_status,0.009840
6,vehicle_type,0.009143
4,day_of_week,0.008500
7,weather,0.008212
17,accident_location_details,0.007709


In [21]:
joblib.dump(preprocessor, "../models/preprocessor.pkl")
joblib.dump(X_train_processed, "../data/processed/X_train.pkl")
joblib.dump(X_test_processed, "../data/processed/X_test.pkl")
joblib.dump(y_train, "../data/processed/y_train.pkl")
joblib.dump(y_test, "../data/processed/y_test.pkl")
joblib.dump(df, "../data/processed/final_dataframe.pkl")
joblib.dump(all_features_transformed, "../models/feature_names.pkl")

['../models/feature_names.pkl']